In [30]:
!pip install -q wikipedia sentence-transformers "tokenizers>=0.22.0,<=0.23.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 9.4 MB/s eta 0:00:00


In [32]:
# ============================================================
#  RAG CHATBOT — Context-Aware with LangChain + FAISS
#  Google Colab Ready  |  Task 4
# ============================================================

import os
import sys
import time
import argparse
import warnings
warnings.filterwarnings("ignore")

# ── CLI flags ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
parser = argparse.ArgumentParser()
parser.add_argument("--build-kb", action="store_true", help="Rebuild KB")
parser.add_argument("--port", type=int, default=8501)
args, unknown = parser.parse_known_args()

# ── Paths ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BASE_DIR   = os.getcwd()
KB_DIR     = os.path.join(BASE_DIR, "kb_faiss")
APP_FILE   = os.path.join(BASE_DIR, "app_ui.py")

# ════════════════════════════════════════════════
#  STEP 1 — BUILD KNOWLEDGE BASE (Using Free Local Embeddings)
# ═══════════════════════════════════════════════╈

def build_knowledge_base():
    print("\n• Building knowledge base using Local HuggingFace Embeddings …")
    from langchain_community.document_loaders import WikipediaLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_community.vectorstores import FAISS
    from langchain_huggingface import HuggingFaceEmbeddings

    TOPICS = ["Artificial intelligence", "Machine learning", "Natural language processing", "Large language model"]
    all_docs = []
    for topic in TOPICS:
        print(f"   ⌇ Loading: {topic} …")
        try:
            loader = WikipediaLoader(query=topic, load_max_docs=1)
            docs = loader.load()
            all_docs.extend(docs)
        except Exception as exc: print(f"      ☐ Skipped ({exc})")

    if not all_docs:
        print("❌ Error: No documents loaded.")
        return

    splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=50)
    chunks = splitter.split_documents(all_docs)

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    db = FAISS.from_documents(chunks, embeddings)
    db.save_local(KB_DIR)
    print(f"   ⌒ Saved FAISS index → {KB_DIR}\n")

# ════════════════════════════════════════════════
#  STEP 2 — APP CODE
# ═══════════════════════════════════════════════╈

APP_CODE = f"""
import os, streamlit as st
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferWindowMemory

KB_DIR = '{KB_DIR}'

st.title("ထ RAG Chatbot (Local Embeddings)")

with st.sidebar:
    user_key = st.text_input("Enter OpenAI API Key", type="password")
    st.info("Knowledge base uses free local embeddings. Only Chat LLM needs a key.")

@st.cache_resource
def load_chain(_api_key):
    embeddings = HuggingFaceEmbeddings(model_name=\"sentence-transformers/all-MiniLM-L6-v2\")
    db = FAISS.load_local(KB_DIR, embeddings, allow_dangerous_deserialization=True)
    retriever = db.as_retriever()
    llm = ChatOpenAI(model_name='gpt-3.5-turbo', temperature=0.3, openai_api_key=_api_key)
    memory = ConversationBufferWindowMemory(k=3, memory_key='chat_history', return_messages=True, output_key='answer')
    return ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

if not os.path.exists(KB_DIR):
    st.error("KB missing. Build it first.")
else:
    if user_key:
        try:
            chain = load_chain(user_key)
            user_input = st.text_input("Ask about AI/ML:")
            if user_input:
                res = chain.invoke({{'question': user_input}})
                st.write("ထ:", res['answer'])
        except Exception as e:
            st.error(f"Error: {{e}}")
    else:
        st.warning("Please enter a valid OpenAI API key in the sidebar.")
"""

def main():
    if args.build_kb or not os.path.exists(KB_DIR):
        build_knowledge_base()
    with open(APP_FILE, "w") as f: f.write(APP_CODE)
    print(f"✅ App UI written → {APP_FILE}")
    print("\nNow run: !streamlit run app_ui.py & npx localtunnel --port 8501")

if __name__ == "__main__":
    main()


• Building knowledge base using Local HuggingFace Embeddings …
   ⌇ Loading: Artificial intelligence …
   ⌇ Loading: Machine learning …
   ⌇ Loading: Natural language processing …
   ⌇ Loading: Large language model …


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

   ⌒ Saved FAISS index → /content/kb_faiss

✅ App UI written → /content/app_ui.py

Now run: !streamlit run app_ui.py & npx localtunnel --port 8501


In [34]:
import urllib
print("Endpoint IP for localtunnel: ", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

# Run streamlit in the background and localtunnel
!nohup streamlit run app_ui.py --server.port 8501 & npx localtunnel --port 8501

Endpoint IP for localtunnel:  35.201.215.41
nohup: appending output to 'nohup.out'
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙your url is: https://four-ghosts-cough.loca.lt
^C


In [27]:
def build_knowledge_base():
    print("\n🔨 Building knowledge base …")
    from langchain_community.document_loaders import WikipediaLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_community.vectorstores import FAISS
    from langchain_openai import OpenAIEmbeddings

    TOPICS = ["Artificial intelligence", "Machine learning", "Natural language processing", "Large language model"]
    all_docs = []
    for topic in TOPICS:
        print(f"   📥 Loading: {topic} …")
        try:
            loader = WikipediaLoader(query=topic, load_max_docs=1)
            docs = loader.load()
            all_docs.extend(docs)
        except Exception as exc:
            print(f"      ⚠️ Skipped ({exc})")
        time.sleep(0.5)

    if not all_docs:
        print("❌ Error: No documents were loaded. FAISS index cannot be created.")
        return

    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    chunks = splitter.split_documents(all_docs)
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=OPENAI_API_KEY)
    db = FAISS.from_documents(chunks, embeddings)
    db.save_local(KB_DIR)
    print(f"   💾 Saved FAISS index → {KB_DIR}\n")